## Wearable Tech: Week 5 Assignment

### Step 1


In [1]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sn
import matplotlib.dates as mdates
import datetime
import plotly.graph_objects as go

import scipy.stats as stats

df = pd.read_csv("/content/drive/MyDrive/Sports Performance Analytics Michigan/Course 4 - Wearable Technologies and Sports Analytics/Week 4 Content/FH.csv")

### Step 2

In [3]:
# Create a copy  of the dataframe and clean up the columns by just focusing on the ones you need.
# Consider making a multi-index with both the timeindex and the AthleteID.
df.drop(['Unnamed: 0', 'Latitude', 'Longitude', 'Heart Rate'], axis=1, inplace=True)
master=df.set_index(['Timestamp', 'AthleteID'], inplace=False)
master.head()

Seconds  Velocity  Acceleration  Odometer  \
Timestamp             AthleteID                                              
9/30/2018 12:21:49 PM Athlete 1      0.0      0.06     -0.041234       0.0   
                      Athlete 1      0.1      0.06     -0.025926       0.0   
                      Athlete 1      0.2      0.06     -0.011945       0.0   
                      Athlete 1      0.3      0.09      0.048539       0.0   
                      Athlete 1      0.4      0.08      0.021406       0.0   

                                 Player Load  
Timestamp             AthleteID               
9/30/2018 12:21:49 PM Athlete 1          0.0  
                      Athlete 1          0.0  
                      Athlete 1          0.0  
                      Athlete 1          0.0  
                      Athlete 1          0.0

### Step 3

In [4]:
# Make your new variables for the performance metric -- specifically the 20 second farthest distance traveled and the 60 second farthest distance traveled.
master['farthest20']=master['Odometer'].diff(200)
master['farthest60']=master['Odometer'].diff(600)

In [5]:
master.loc[master['farthest20'] <0,'farthest20'] = np.nan
master.loc[master['farthest60'] <0,'farthest60'] = np.nan

### Step 4

In [6]:
# Run a groupby with agg to collect your max values for acceleration and your new variables.
MaxValues_df=master.groupby('AthleteID').agg([max])
print(MaxValues_df)

/tmp/ipykernel_215/575283181.py:2: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  MaxValues_df=master.groupby('AthleteID').agg([max])
/tmp/ipykernel_215/575283181.py:2: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  MaxValues_df=master.groupby('AthleteID').agg([max])


            Seconds Velocity Acceleration  Odometer Player Load farthest20  \
                max      max          max       max         max        max   
AthleteID                                                                    
Athlete 1   8759.98     7.24     5.981192  10036.75       891.5     119.37   
Athlete 10  8749.93     6.79     5.580821   7269.15       615.6     111.93   
Athlete 11  8749.93     6.56     3.825559   4791.84       449.0      87.14   
Athlete 12  8749.91     7.38     4.360430   8551.21       734.4     177.37   
Athlete 13  8749.93     6.33     7.034368   6345.99       609.2      83.81   
Athlete 14  7855.38     6.58     4.908402   9906.41       951.3      94.24   
Athlete 15  8749.93     6.61    11.318196   5001.16       540.0      85.37   
Athlete 17  8749.93     7.71     5.889059   9896.45       888.5      97.05   
Athlete 18  8759.92     6.63     5.557953   6326.56       688.5      81.20   
Athlete 19  8749.92     6.85     4.089531   7952.18       781.6 

### Step 5

In [7]:
# Calculate the z-scores for all of your performance variables.
MaxValues_df['zscores_acc'] = stats.zscore(MaxValues_df['Acceleration'])
MaxValues_df['zscores_far20'] = stats.zscore(MaxValues_df['farthest20'])
MaxValues_df['zscores_far60'] = stats.zscore(MaxValues_df['farthest60'])
print(MaxValues_df)

            Seconds Velocity Acceleration  Odometer Player Load farthest20  \
                max      max          max       max         max        max   
AthleteID                                                                    
Athlete 1   8759.98     7.24     5.981192  10036.75       891.5     119.37   
Athlete 10  8749.93     6.79     5.580821   7269.15       615.6     111.93   
Athlete 11  8749.93     6.56     3.825559   4791.84       449.0      87.14   
Athlete 12  8749.91     7.38     4.360430   8551.21       734.4     177.37   
Athlete 13  8749.93     6.33     7.034368   6345.99       609.2      83.81   
Athlete 14  7855.38     6.58     4.908402   9906.41       951.3      94.24   
Athlete 15  8749.93     6.61    11.318196   5001.16       540.0      85.37   
Athlete 17  8749.93     7.71     5.889059   9896.45       888.5      97.05   
Athlete 18  8759.92     6.63     5.557953   6326.56       688.5      81.20   
Athlete 19  8749.92     6.85     4.089531   7952.18       781.6 

### Step 6

In [8]:
# Using the z-scores, sum the values to determine the performance scores and then add 10 to the summed values.
MaxValues_df['Metric'] = stats.zscore(MaxValues_df['Acceleration']) + stats.zscore(MaxValues_df['farthest20'])+stats.zscore(MaxValues_df['farthest60']) + 10
print(MaxValues_df)

            Seconds Velocity Acceleration  Odometer Player Load farthest20  \
                max      max          max       max         max        max   
AthleteID                                                                    
Athlete 1   8759.98     7.24     5.981192  10036.75       891.5     119.37   
Athlete 10  8749.93     6.79     5.580821   7269.15       615.6     111.93   
Athlete 11  8749.93     6.56     3.825559   4791.84       449.0      87.14   
Athlete 12  8749.91     7.38     4.360430   8551.21       734.4     177.37   
Athlete 13  8749.93     6.33     7.034368   6345.99       609.2      83.81   
Athlete 14  7855.38     6.58     4.908402   9906.41       951.3      94.24   
Athlete 15  8749.93     6.61    11.318196   5001.16       540.0      85.37   
Athlete 17  8749.93     7.71     5.889059   9896.45       888.5      97.05   
Athlete 18  8759.92     6.63     5.557953   6326.56       688.5      81.20   
Athlete 19  8749.92     6.85     4.089531   7952.18       781.6 

### Step 7

**Finally, evaluate whether the same top 2 athletes are still on top of the rankings even though the criteria changed.**

In the workbook from the previous session our top 2 athletes in terms of our Metric were athlete 1 and athlete 12.  Looking at the new calculation for our Metric our top 2 athletes are athlete 10 and athlete 12.  So athlete 12 is a constant while athlete 1 got taken over by athlete 10 using the new calculations of our global metric.